# 面试问题：什么时候需要 Multi-Agent，Agent Handoff、共享状态和冲突怎样设计？

**一句话回答**：只有任务确实需要不同能力/权限、动态并行或独立审查，且收益超过额外 token 与协调风险时才用多 Agent。每次委派传递最小 task contract、deadline、权限和返回 schema；子 Agent 不继承全部上下文/凭证。共享 blackboard 用来源、租户和版本控制，handoff graph 限制深度/环；冲突由确定性聚合或人工解决。

本 Notebook 实现能力注册表、最小委派包、路由、CAS blackboard、委派图、冲突聚合、故障隔离和系统级评测。

In [ ]:
from dataclasses import dataclass
from collections import defaultdict
import hashlib, json, math

SEED121=12101
assert SEED121==12101
assert hashlib.sha256(b"agent-a").hexdigest()!=hashlib.sha256(b"agent-b").hexdigest()
assert defaultdict(list)["x"]==[]

## 1. Agent 注册表描述能力，也限制权限

Agent 声明 capability、允许工具、数据域、单位成本和并发上限。描述不是自我证明，发布前由平台评测与签名。专用 Agent 只获得完成任务所需 scope；“万能 supervisor”不应默认拥有所有下游凭证。

In [ ]:
@dataclass(frozen=True)
class Agent121:
    agent_id:str; capabilities:frozenset; scopes:frozenset; tenant:str; cost:float; trusted:bool=True
    def __post_init__(self):
        if not self.agent_id or self.cost<0 or not self.capabilities: raise ValueError("agent_contract")
agents121=[Agent121("research",frozenset({"search","summarize"}),frozenset({"web:read"}),"T1",.02),Agent121("finance",frozenset({"calculate","risk"}),frozenset({"ledger:read"}),"T1",.03),Agent121("writer",frozenset({"compose"}),frozenset(),"T1",.01)]
assert len({a.agent_id for a in agents121})==3
assert all(a.trusted for a in agents121)
try: Agent121("",frozenset(),frozenset(),"T",-1); raise AssertionError("bad agent accepted")
except ValueError as e: assert str(e)=="agent_contract"

## 2. Handoff 是结构化任务，不是转发整段聊天

委派包包含 task ID、目标、required capability、输入引用、允许 scope、deadline、预算、返回 schema 和 parent trace。只发送必要上下文；用户私密历史、其他租户证据和 supervisor 的系统指令不应自动下放。

In [ ]:
@dataclass(frozen=True)
class Handoff121:
    task_id:str; goal:str; required:frozenset; allowed_scopes:frozenset; context_refs:tuple; parent:str; depth:int; budget:float
    def __post_init__(self):
        if not self.task_id or not self.goal or self.depth<0 or self.budget<=0: raise ValueError("handoff_contract")
handoff121=Handoff121("h1","搜索市场资料",frozenset({"search"}),frozenset({"web:read"}),("doc:user-query",),"root",1,.05)
assert handoff121.required==frozenset({"search"})
assert handoff121.context_refs==("doc:user-query",)
try: Handoff121("","x",frozenset(),frozenset(),(),"r",-1,0); raise AssertionError("bad handoff accepted")
except ValueError as e: assert str(e)=="handoff_contract"

## 3. 路由先满足 capability/scope/tenant，再优化成本

不能仅按自然语言相似度挑 Agent。硬过滤 required capability、allowed scope、tenant、健康和预算；多个可行者再按校准成功率、负载和成本排序。无可行 Agent 时返回无法委派，而不是扩大权限。

In [ ]:
success121={"research":.9,"finance":.86,"writer":.95}
def select_agent121(h,tenant):
    feasible=[a for a in agents121 if a.trusted and a.tenant==tenant and h.required<=a.capabilities and a.scopes<=h.allowed_scopes and a.cost<=h.budget]
    return min(feasible,key=lambda a:a.cost/max(success121[a.agent_id],1e-9)) if feasible else None
selected121=select_agent121(handoff121,"T1")
assert selected121.agent_id=="research"
assert select_agent121(handoff121,"T2") is None
assert select_agent121(Handoff121("h2","x",frozenset({"admin"}),frozenset(),(),"root",1,.1),"T1") is None

## 4. Context isolation 与 provenance 一起传递

子 Agent 通过受控引用读取内容，读取时再次 ACL；输出标明输入引用、模型/Agent 版本和 taint。peer message 是不可信数据，不能成为更高优先级指令。聚合器只接受声明 schema，拒绝子 Agent 夹带新工具请求。

In [ ]:
resources121={"doc:user-query":{"tenant":"T1","text":"分析市场","sensitive":False},"doc:secret":{"tenant":"T2","text":"秘密","sensitive":True}}
def resolve_context121(refs,tenant):
    out=[]
    for r in refs:
        if r not in resources121 or resources121[r]["tenant"]!=tenant: raise ValueError("context_acl")
        out.append({"ref":r,"text":resources121[r]["text"],"untrusted":True})
    return out
ctx121=resolve_context121(handoff121.context_refs,"T1")
assert ctx121==[{"ref":"doc:user-query","text":"分析市场","untrusted":True}]
try: resolve_context121(("doc:secret",),"T1"); raise AssertionError("cross tenant accepted")
except ValueError as e: assert str(e)=="context_acl"
assert all(x["untrusted"] for x in ctx121)

## 5. Blackboard 使用不可变条目与乐观版本

Agent 不直接覆盖彼此结论，而是追加带 author/source/confidence 的候选；若要更新共享任务状态，用 expected version CAS。这样可追踪冲突与恶意/过期写入。敏感信息按字段 ACL，不把整个 blackboard 广播给所有 Agent。

In [ ]:
board121={"version":0,"entries":[]}
def board_append121(expected,entry):
    if expected!=board121["version"]: return False,board121["version"]
    required={"key","value","author","source"}
    if set(entry)!=required: raise ValueError("entry_schema")
    board121["entries"].append(dict(entry)); board121["version"]+=1; return True,board121["version"]
first121=board_append121(0,{"key":"risk","value":"low","author":"finance","source":"ledger:v3"}); stale121=board_append121(0,{"key":"risk","value":"high","author":"research","source":"web:d1"})
assert first121==(True,1) and stale121==(False,1)
assert board121["entries"][0]["author"]=="finance"
assert board121["version"]==1

## 6. 委派图限制深度、fan-out、环和总预算

Agent A→B→A 会无限循环；每个 handoff 继承 visited path 和剩余预算。达到最大深度、重复 Agent、fan-out 或预算不足立即停止。子 Agent 不得自行突破 supervisor 的总配额。

In [ ]:
def can_delegate121(path,target,depth,max_depth,remaining,cost):
    if depth>=max_depth: return False,"depth"
    if target in path: return False,"cycle"
    if cost>remaining: return False,"budget"
    return True,"allowed"
assert can_delegate121(("root",),"research",1,3,.1,.02)==(True,"allowed")
assert can_delegate121(("root","research"),"root",2,3,.1,.01)==(False,"cycle")
assert can_delegate121(("root","research","finance"),"writer",3,3,.1,.01)==(False,"depth")

## 7. 冲突不能靠“多数 Agent”自动变真

独立证据可按权威/新鲜度聚合；相同来源复制出的多个 Agent 不算独立票。事实冲突保留两边 provenance，按领域规则或人工裁决。写操作只有单一 commit coordinator，避免多个 Agent 各自提交。

In [ ]:
claims121=[{"value":"low","source":"ledger:v3","authority":3,"time":10},{"value":"high","source":"web:d1","authority":1,"time":20},{"value":"low","source":"ledger:v3","authority":3,"time":10}]
def merge_claims121(claims):
    unique={(c["source"],c["value"]):c for c in claims}; ranked=sorted(unique.values(),key=lambda c:(-c["authority"],-c["time"],c["source"])); top=ranked[0]; conflicts=[c for c in ranked[1:] if c["value"]!=top["value"]]; return top,conflicts
top121,conflicts121=merge_claims121(claims121)
assert top121["value"]=="low" and top121["source"]=="ledger:v3"
assert len(conflicts121)==1 and conflicts121[0]["value"]=="high"
assert len({(c["source"],c["value"]) for c in claims121})==2

## 8. 系统评测要与单 Agent baseline 做 ablation

报 task success、安全违规、总 token、墙钟、handoff 数、冲突率、无效委派和人工升级；分别移除 specialist、共享板或并行，确认收益来自哪里。一个模型扮演多个角色并不自动产生独立性，多 Agent 可能只增加相关错误与成本。

In [ ]:
results121=[{"arch":"single","success":.72,"cost":.03,"violations":.01},{"arch":"multi","success":.84,"cost":.09,"violations":.008}]; utility121=lambda r:r["success"]-r["cost"]-5*r["violations"]
manifest121={"schema":1,"registry":"agents-v3","handoff":"minimal-context-v2","max_depth":3,"max_fanout":4,"board":"cas+provenance","commit":"single_coordinator"}; digest121=hashlib.sha256(json.dumps(manifest121,sort_keys=True).encode()).hexdigest()
assert utility121(results121[1])>utility121(results121[0])
assert manifest121["max_depth"]==3 and manifest121["commit"]=="single_coordinator"
assert len(digest121)==64

## 面试总结

回答结构是：**证明需要多 Agent → 治理能力注册表 → 最小 handoff 合同 → capability/scope 路由 → context/credential 隔离 → provenance+CAS blackboard → 深度/环/预算 → 冲突与单点提交 → 系统级 ablation**。多 Agent 是分布式系统问题，不是多写几个角色 Prompt。

延伸阅读：[AutoGen](https://arxiv.org/abs/2308.08155)、[Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents)、[OWASP Agentic Threats Navigator](https://genai.owasp.org/resource/owasp-gen-ai-security-project-agentic-threats-navigator/)。